# Enc — Encoding comparison, all five models, CD-HIT-97%, ORIGINAL encoders and architectures

**Why this notebook exists, and why it changed from the previous draft.** The previous Enc used
a from-scratch TF-IDF/Word2Vec/hybrid implementation and the simplified CNN architectures. Your
three original per-architecture notebooks (the ones that first ran and recorded the encoding-
comparison scores behind the old Table 6/7) use their own specific encoder implementations and
the ORIGINAL (heavier) CNN architectures -- see Fragsweep/Ambig for the full list of architecture
differences this corrects. This version uses the exact original encoders and architectures,
changing only: (1) the split -- CD-HIT-97% instead of MinHash, and (2) the model roster -- adds
the linear SVM and random forest to the full encoding grid (closing the gap your reviewers
flagged: the SVM was never run through the encoding comparison before), and (3) drops the
CGR-TFIDF-6mer encoding entirely, per your decision to keep the paper's scope to four encodings
across exactly five models.

**Encodings kept (verbatim from the original notebooks' own encoder functions):**
- **6mer**: character 6-mers, uncapped vocabulary (every distinct 6-mer seen in training), raw
  frequency normalized so each sequence's vector sums to 1. This is NOT the same as the capped,
  512-feature "raw 6-mer"/Kameris-equivalent encoding used elsewhere in the project -- it is its
  own thing, named "6mer" in your original notebooks, and worth keeping distinct in the writeup.
- **TF-IDF-6mer**: `TfidfVectorizer` over space-joined 6-mer tokens, top 512 features. (Mechanically
  equivalent to the char-n-gram TF-IDF convention used in T1/T4/Fragsweep/Ambig, just tokenized
  differently -- should produce effectively the same features.)
- **word2vec6mer**: a Word2Vec model (vector_size=100, window=5, min_count=1, **seed=42 fixed**,
  10 epochs -- fixed seed exactly as in your original notebooks, not varied per experiment seed)
  trained on training-set 6-mers; each sequence's vector is the mean of its 6-mers' embeddings.
- **Word2Vec-TF-IDF-6mer** (hybrid): Word2Vec vector concatenated with the TF-IDF-6mer vector, in
  that order, exactly as in the original `make_w2v_tfidf_encoder`.

**GPU needed.** Yes, for the CNN half. Budget similarly to Fragsweep/Ambig -- these are the same heavy
architectures, so each CNN fit is roughly 15-30 minutes at full length, likely longer than the
2-8 minutes the simplified architecture needed. Your original notebooks took 5-8 hours per
architecture for the 3-seed x 5-encoding grid (one architecture at a time); this notebook runs
all three architectures together in one pass per seed/encoding, so expect a similar total.

**Resume-safe**, same convention as Fragsweep/Ambig: it loads any existing
`enc_encoding_all5_raw.csv` and skips (model, encoding, seed) combinations already done.
**`FAST_MODE = True` (default) runs 2 seeds instead of 3** to shrink the first pass; expand once
you've confirmed it runs cleanly and you have quota to spare.

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


**Editorial note (clean repo build).** The `build_resnet18` cell below has had its `GaussianNoise(0.05)` input layer removed, matching the final no-noise architecture used throughout this study. SVM, RF, VGG-16 and Inception-v1 were never affected. `encoding_all5_classical_vgg_inception_and_legacy_resnet_raw.csv` (this notebook's own output) still contains the old, superseded GaussianNoise-era ResNet-18 rows alongside the unaffected and still-authoritative SVM/RF/VGG-16/Inception-v1 rows; the authoritative no-noise ResNet-18 numbers are in `resnet18_encoding_nonoise_raw.csv` / `_summary.csv`. GaussianNoise removed from 1 cell(s).

### 1. Imports

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os
import numpy as np, pandas as pd
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, balanced_accuracy_score
from gensim.models import Word2Vec

from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, ReLU, Add,
    GlobalAveragePooling1D, Dropout, Dense, MaxPooling1D, GaussianNoise,
    SpatialDropout1D, Layer, Concatenate)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

import shutil, subprocess
OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

if shutil.which("cd-hit-est") is None:
    print("cd-hit-est not found, installing...")
    subprocess.run("conda install -y -c bioconda cd-hit 2>/dev/null || true", shell=True)
    if shutil.which("cd-hit-est") is None:
        subprocess.run(
            "wget -q https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz "
            "&& tar xzf cd-hit-v4.8.1-2019-0228.tar.gz && cd cd-hit-v4.8.1-2019-0228 && make -j4 "
            "&& cp cd-hit-est /usr/local/bin/ && cp cd-hit /usr/local/bin/", shell=True)
print("cd-hit-est available:", shutil.which("cd-hit-est") is not None)


### 2. Config

In [ ]:

FAST_MODE = False
SEEDS = [0, 1] if FAST_MODE else [0, 1, 2]
ENCODINGS = ["6mer", "TF-IDF-6mer", "word2vec6mer", "Word2Vec-TF-IDF-6mer"]   # CGR dropped
VEC_SIZE = 100
IDENTITY = 0.97
EPOCHS = 120
BATCH  = 48


### 3. Load data and build the CD-HIT-97% grouping

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)
le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
n = len(df)
print(f"{n} sequences, {n_cls} classes: {CLASSES}")

with open(f"{OUT}/pol_enc.fasta", "w") as f:
    for i, s in enumerate(X_all):
        f.write(f">{i}\n{s}\n")

cmd = f"cd-hit-est -i {OUT}/pol_enc.fasta -o {OUT}/pol_enc_cdhit97 -c {IDENTITY} -n 10 -M 16000 -T 4 -d 0"
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
assert r.returncode == 0, r.stderr

def parse_cdhit_clusters(clstr_path, n_seqs):
    labels = np.full(n_seqs, -1, dtype=int)
    cid = -1
    with open(clstr_path) as f:
        for line in f:
            if line.startswith(">Cluster"):
                cid += 1
            else:
                idx = int(line.split(">")[1].split("...")[0])
                labels[idx] = cid
    assert (labels >= 0).all()
    return labels

grp_all = parse_cdhit_clusters(f"{OUT}/pol_enc_cdhit97.clstr", n)
print(f"CD-HIT 97% identity: {len(set(grp_all))} clusters")


### 4. Split helper

In [ ]:

def split_grouped(X, y, grp, seed):
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te


### 5. The ORIGINAL encoders (copied verbatim from your three per-architecture notebooks)

In [ ]:

def kmers(seq, k): return [seq[i:i+k] for i in range(len(seq) - k + 1)]

def make_kmer_encoder(train, k):
    vocab = sorted({km for s in train for km in kmers(s, k)})
    idx = {kk: i for i, kk in enumerate(vocab)}
    def tf_(seqs):
        out = np.zeros((len(seqs), len(vocab)), np.float32)
        for i, s in enumerate(seqs):
            for km in kmers(s, k):
                if km in idx: out[i, idx[km]] += 1
            tot = out[i].sum()
            if tot > 0: out[i] /= tot
        return out
    return tf_

def make_w2v_encoder(train, k, vs=VEC_SIZE):
    mdl = Word2Vec([kmers(s, k) for s in train], vector_size=vs, window=5,
                   min_count=1, workers=4, seed=42, epochs=10)
    def tf_(seqs):
        out = []
        for s in seqs:
            v = [mdl.wv[w] for w in kmers(s, k) if w in mdl.wv]
            out.append(np.mean(v, 0) if v else np.zeros(vs))
        return np.array(out, np.float32)
    return tf_

def make_tfidf_encoder(train, k, max_features=512):
    vec = TfidfVectorizer(max_features=max_features)
    vec.fit([" ".join(kmers(s, k)) for s in train])
    return lambda seqs: vec.transform(
        [" ".join(kmers(s, k)) for s in seqs]).toarray().astype(np.float32)

def make_w2v_tfidf_encoder(train, k):
    w = make_w2v_encoder(train, k); t = make_tfidf_encoder(train, k)
    return lambda seqs: np.concatenate([w(seqs), t(seqs)], axis=1)

def build_encoders(X_tr, names):
    reg = {
        "6mer":                 lambda: make_kmer_encoder(X_tr, 6),
        "TF-IDF-6mer":          lambda: make_tfidf_encoder(X_tr, 6),
        "word2vec6mer":         lambda: make_w2v_encoder(X_tr, 6),
        "Word2Vec-TF-IDF-6mer": lambda: make_w2v_tfidf_encoder(X_tr, 6),
    }
    return {n: reg[n]() for n in names}


### 6. The ORIGINAL CNN architectures (same as Fragsweep/Ambig -- copied verbatim)

In [ ]:

class StochasticDepth(Layer):
    def __init__(self, p=0.85, **kw): super().__init__(**kw); self.p = p
    def call(self, x, training=None):
        if not training: return x
        r = tf.random.uniform([tf.shape(x)[0], 1, 1])
        return x * tf.floor(self.p + r) / self.p
    def get_config(self): return {**super().get_config(), "p": self.p}

def res_block(x, f, stride=1, sp=0.85, dr=0.1):
    sc = x
    x = Conv1D(f, 3, strides=stride, padding="same", use_bias=False)(x)
    x = ReLU()(BatchNormalization()(x)); x = SpatialDropout1D(dr)(x)
    x = BatchNormalization()(Conv1D(f, 3, padding="same", use_bias=False)(x))
    if stride != 1 or sc.shape[-1] != f:
        sc = BatchNormalization()(Conv1D(f, 1, strides=stride, padding="same", use_bias=False)(sc))
    return ReLU()(Add()([StochasticDepth(sp)(x), sc]))

def build_resnet18(input_len, n_classes):
    inp = Input((input_len, 1)); x = inp  # GaussianNoise REMOVED -- final architecture used throughout this study has no input-noise augmentation
    x = ReLU()(BatchNormalization()(Conv1D(32, 7, strides=2, padding="same", use_bias=False)(x)))
    x = MaxPooling1D(3, strides=2, padding="same")(x)
    for f, sp in [(32, .95), (32, .92)]:  x = res_block(x, f, 1, sp)
    for f, sp in [(64, .88), (64, .85)]:  x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    for f, sp in [(128, .80), (128, .78)]: x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    for f, sp in [(256, .75), (256, .72)]: x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def build_vgg16(input_len, n_classes):
    def conv_block(x, f, n_convs):
        for _ in range(n_convs):
            x = ReLU()(BatchNormalization()(Conv1D(f, 3, padding="same", use_bias=False)(x)))
        return MaxPooling1D(2, padding="same")(x)
    inp = Input((input_len, 1))
    x = conv_block(inp, 32, 2)
    x = conv_block(x,   64, 2)
    x = conv_block(x,  128, 3)
    x = conv_block(x,  256, 3)
    x = conv_block(x,  256, 3)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    x = Dropout(0.4)(ReLU()(Dense(256)(x)))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def inception_module(x, f1, f3r, f3, f5r, f5, fp):
    b1 = Conv1D(f1, 1, padding="same", activation="relu")(x)
    b3 = Conv1D(f3, 3, padding="same", activation="relu")(Conv1D(f3r, 1, padding="same", activation="relu")(x))
    b5 = Conv1D(f5, 5, padding="same", activation="relu")(Conv1D(f5r, 1, padding="same", activation="relu")(x))
    bp = Conv1D(fp, 1, padding="same", activation="relu")(MaxPooling1D(3, strides=1, padding="same")(x))
    return Concatenate()([b1, b3, b5, bp])

def build_inception(input_len, n_classes):
    inp = Input((input_len, 1))
    x = MaxPooling1D(3, strides=2, padding="same")(
            ReLU()(BatchNormalization()(Conv1D(32, 7, strides=2, padding="same")(inp))))
    x = inception_module(x, 16, 24, 32, 4,  8,  8)
    x = inception_module(x, 32, 32, 48, 8, 24, 16)
    x = MaxPooling1D(3, strides=2, padding="same")(x)
    x = inception_module(x, 48, 24, 52, 8, 24, 16)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

BUILDERS = {"resnet18": build_resnet18, "vgg16": build_vgg16, "inception": build_inception}

def run_deep_original(model_name, Xtr, ytr, Xva, yva, Xte, yte, seed, n_classes):
    tf.random.set_seed(seed)
    sc = StandardScaler()
    Xtr_d = sc.fit_transform(np.asarray(Xtr, dtype=np.float32))[..., None]
    Xva_d = sc.transform(np.asarray(Xva, dtype=np.float32))[..., None]
    Xte_d = sc.transform(np.asarray(Xte, dtype=np.float32))[..., None]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))
    model = BUILDERS[model_name](Xtr_d.shape[1], n_classes)
    model.fit(Xtr_d, to_categorical(ytr, n_classes),
              validation_data=(Xva_d, to_categorical(yva, n_classes)),
              epochs=EPOCHS, batch_size=BATCH, class_weight=cw, shuffle=True, verbose=0,
              callbacks=[EarlyStopping("val_loss", patience=15, restore_best_weights=True),
                         ReduceLROnPlateau("val_loss", factor=0.5, patience=6, min_lr=1e-6)])
    prob = model.predict(Xte_d, verbose=0)
    pred = prob.argmax(1)
    f1 = f1_score(yte, pred, average="macro"); bal = balanced_accuracy_score(yte, pred)
    tf.keras.backend.clear_session()
    return f1, bal


### 7. Resume support

In [ ]:

RAW_PATH = f"{OUT}/enc_encoding_all5_raw.csv"
if os.path.exists(RAW_PATH):
    existing = pd.read_csv(RAW_PATH)
    done_keys = set(zip(existing["model"], existing["encoding"], existing["seed"]))
    rows = existing.to_dict("records")
    print(f"Resuming: {len(existing)} existing rows, {len(done_keys)} combos already done.")
else:
    rows = []
    done_keys = set()
    print("No existing results -- starting fresh.")


### 8. Main sweep: (encoding x seed x model), full-length sequences, CD-HIT-97% split

In [ ]:

for seed in SEEDS:
    tr, va, te = split_grouped(X_all, y_all, grp_all, seed)
    ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
    seq_tr, seq_va, seq_te = X_all[tr], X_all[va], X_all[te]

    all_models = ["linear_svm", "random_forest", "resnet18", "vgg16", "inception"]
    if all((m, enc, seed) in done_keys for enc in ENCODINGS for m in all_models):
        print(f"seed={seed} already fully done, skipping")
        continue

    encoders = build_encoders(seq_tr, ENCODINGS)

    for enc_name in ENCODINGS:
        if all((m, enc_name, seed) in done_keys for m in all_models):
            continue
        enc = encoders[enc_name]
        Xtr_raw, Xva_raw, Xte_raw = enc(seq_tr), enc(seq_va), enc(seq_te)

        for model_name, clf in [
            ("linear_svm", LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000)),
            ("random_forest", RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                                        random_state=seed, n_jobs=-1)),
        ]:
            if (model_name, enc_name, seed) in done_keys:
                continue
            clf.fit(Xtr_raw, ytr)
            pred = clf.predict(Xte_raw)
            rows.append(dict(model=model_name, encoding=enc_name, seed=seed,
                              f1=f1_score(yte, pred, average="macro"),
                              bal_acc=balanced_accuracy_score(yte, pred)))

        for model_name in ["resnet18", "vgg16", "inception"]:
            if (model_name, enc_name, seed) in done_keys:
                continue
            f1, bal = run_deep_original(model_name, Xtr_raw, ytr, Xva_raw, yva, Xte_raw, yte, seed, n_cls)
            rows.append(dict(model=model_name, encoding=enc_name, seed=seed, f1=f1, bal_acc=bal))

        pd.DataFrame(rows).to_csv(RAW_PATH, index=False)  # incremental save
        print(f"seed={seed} encoding={enc_name} done ({len(rows)} rows so far)")


### 9. Summary table -- this is what the new Table 6/7 gets rebuilt from

In [ ]:

raw = pd.DataFrame(rows)
summ = raw.groupby(["model","encoding"])[["f1","bal_acc"]].agg(["mean","std"]).round(4)
summ.columns = [f"{a}_{b}" for a,b in summ.columns]
summ = summ.reset_index()
summ.to_csv(f"{OUT}/enc_encoding_all5_summary.csv", index=False)
print(summ.pivot(index="model", columns="encoding", values="f1_mean").round(4))


### 10. Zip outputs

In [ ]:
!cd /kaggle/working && zip -r enc_encoding_all5_outputs.zip enc_encoding_all5_raw.csv enc_encoding_all5_summary.csv